In [1]:
# Parameters
run_id = "130b4b0c-059c-4a33-880b-6e04c41ce18c"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/130b4b0c-059c-4a33-880b-6e04c41ce18c"
sample_size = None
epochs = None
threshold = None


### Model training 
In the previous notebook we performed hyperparamer tuning. Now we are ready to train the embeddings model based on the best hyper parameters and export to model repository.
![Training Dataset](./images/experiment_td.png)

Here as well we are going to use StellarGraph library to compute node embeddings. StellarGraph supports loading data via Pandas DataFrames, NumPy arrays, Neo4j and NetworkX graphs. 

---
**NOTE**:

Loading large scale dataset in to StellarGraph for training can not be handled with above mentioned fameworks. It will require loading data using frameworks such as `tf.data`. 

If your training datasets measure from couple of GB to 100s of GBs or even TBs contact us at Logical Clocks and we will help you to setup distributed training pipelines. 

---

### Define hopsworks experiments wrapper function and put all the training logic there. 

In [2]:
# Imports and setup for local execution
import os
import json
import uuid
import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")
MODELS_PATH = os.path.join(BASE_PATH, "models")
os.makedirs(MODELS_PATH, exist_ok=True)

print(f"Training data path: {TRAINING_DATA_PATH}")
print(f"Models will be saved to: {MODELS_PATH}")

Training data path: /home/adnoman/projects/aml_gan/AMLend2end/training_data
Models will be saved to: /home/adnoman/projects/aml_gan/AMLend2end/models


## Use above experiments wrapper function to conduct hops training experiments.

In [3]:
# Load best hyperparameters from previous notebook
best_hyperparams_path = os.path.join(RESOURCES_PATH, "embeddings_best_hp.json")

with open(best_hyperparams_path, 'r') as f:
    best_hyperparams = json.load(f)

print(f"Loaded best hyperparameters: {best_hyperparams}")

Loaded best hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}


In [4]:
# Load training data
print("Loading training data...")
node_pdf = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "node_td.csv"))
edge_pdf = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "edges_td.csv"))
alert_nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv"))

print(f"Nodes: {len(node_pdf)}, Edges: {len(edge_pdf)}, Alert nodes: {len(alert_nodes_df)}")

# Build NetworkX graph
print("\nBuilding NetworkX directed graph...")
G = nx.from_pandas_edgelist(
    edge_pdf, 
    source='source', 
    target='target', 
    edge_attr=['tx_type', 'base_amt'],
    create_using=nx.DiGraph()
)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Loading training data...
Nodes: 7347, Edges: 438386, Alert nodes: 7347

Building NetworkX directed graph...


Graph: 7347 nodes, 17070 edges


In [5]:
# Train Node2Vec model with best hyperparameters
walk_number = best_hyperparams['walk_number']
walk_length = best_hyperparams['walk_length']
emb_size = best_hyperparams['emb_size']

print(f"Training Node2Vec with: walk_number={walk_number}, walk_length={walk_length}, emb_size={emb_size}")

# Create and train Node2Vec model
node2vec = Node2Vec(
    G, 
    dimensions=emb_size, 
    walk_length=walk_length, 
    num_walks=walk_number,
    p=0.5,
    q=2.0,
    workers=4,
    quiet=False
)

print("\nTraining Word2Vec on random walks...")
model = node2vec.fit(window=10, min_count=1, batch_words=4)
print("Training complete!")

Training Node2Vec with: walk_number=2, walk_length=2, emb_size=32


Computing transition probabilities:   0%|          | 0/7347 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 1/1 [00:00<00:00, 71.12it/s]


Generating walks (CPU: 2): 100%|██████████| 1/1 [00:00<00:00, 79.60it/s]
Generating walks (CPU: 3): 0it [00:00, ?it/s]


Generating walks (CPU: 4): 0it [00:00, ?it/s]



Training Word2Vec on random walks...


Training complete!


### Managing experiments
Experiments service provides a unified view of all the experiments run using the `experiment` module.
<br>
As demonstrated in the gif it provides general information about the experiment and the resulting metric. Experiments can be visualized meanwhile or after training in a TensorBoard.
<br>
<br>
![Image7-Monitor.png](./images/experiments.gif)

In [6]:
# Extract embeddings for all nodes
print("Extracting node embeddings...")

embeddings_dict = {}
for node in G.nodes():
    node_str = str(node)
    if node_str in model.wv:
        embeddings_dict[node_str] = model.wv[node_str]

print(f"Generated embeddings for {len(embeddings_dict)} nodes")

# Create embeddings dataframe
embeddings_df = pd.DataFrame.from_dict(embeddings_dict, orient='index')
embeddings_df.index.name = 'node_id'
embeddings_df.columns = [f'emb_{i}' for i in range(emb_size)]
embeddings_df = embeddings_df.reset_index()

print(f"\nEmbeddings shape: {embeddings_df.shape}")
embeddings_df.head()

Extracting node embeddings...
Generated embeddings for 7347 nodes

Embeddings shape: (7347, 33)


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31
0,3aa9646b,0.006354,-0.020029,0.000121,0.001682,0.022640,0.022485,-0.022633,-0.018184,-0.013343,...,-0.005648,0.013926,0.027505,-0.010198,0.025302,-0.008416,0.009119,0.013350,0.022079,-0.029613
1,1e46e726,0.002481,-0.007734,0.013989,-0.021754,0.023876,0.024569,0.005226,-0.014404,-0.014850,...,0.009040,-0.014103,0.029731,-0.024120,0.001362,0.002191,-0.000155,-0.023380,0.027062,0.014415
2,49203bc3,0.024728,-0.005727,0.011389,-0.003658,0.005619,0.027766,0.028874,-0.025506,-0.024554,...,-0.026760,-0.002589,0.025417,0.011257,-0.031272,0.029479,0.019596,-0.007734,0.025463,0.013300
3,a74d1101,0.006074,-0.028726,-0.022943,0.030213,0.001226,-0.016350,-0.007838,-0.003526,-0.005832,...,-0.024862,-0.011728,-0.017198,-0.015983,0.005139,0.021491,-0.009442,0.014519,-0.013948,-0.029704
4,616d4505,-0.020141,-0.024803,-0.019287,-0.006866,-0.015812,0.008584,-0.000066,0.009143,-0.012181,...,-0.030604,-0.023667,0.026482,-0.003340,-0.031200,-0.025585,0.000395,-0.019562,-0.010542,0.011896


In [7]:
# Evaluate embeddings quality using is_sar classification
print("Evaluating embeddings quality...")

# Get embeddings for nodes in alert_nodes_df
X = []
y = []
for _, row in alert_nodes_df.iterrows():
    node_id = str(row['id'])
    if node_id in embeddings_dict:
        X.append(embeddings_dict[node_id])
        y.append(row['is_sar'])

X = np.array(X)
y = np.array(y)

print(f"Evaluation dataset: {len(X)} nodes")

# Train/test split and evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nEmbedding Evaluation Accuracy: {accuracy:.4f}")
metrics = {'accuracy': accuracy}

Evaluating embeddings quality...
Evaluation dataset: 7347 nodes



Embedding Evaluation Accuracy: 0.8891


In [8]:
# Save model and embeddings locally (replaces Hopsworks model registry)
model_id = str(uuid.uuid4())[:8]
model_dir = os.path.join(MODELS_PATH, f"node_embeddings_{model_id}")
os.makedirs(model_dir, exist_ok=True)

# Save Word2Vec model
model_path = os.path.join(model_dir, "node2vec_model.model")
model.save(model_path)
print(f"Saved Node2Vec model to: {model_path}")

# Save classifier
clf_path = os.path.join(model_dir, "classifier.joblib")
joblib.dump(clf, clf_path)
print(f"Saved classifier to: {clf_path}")

# Save embeddings as CSV
embeddings_path = os.path.join(model_dir, "node_embeddings.csv")
embeddings_df.to_csv(embeddings_path, index=False)
print(f"Saved embeddings to: {embeddings_path}")

# Save metrics and hyperparameters
metadata = {
    'hyperparameters': best_hyperparams,
    'metrics': metrics,
    'num_nodes': len(embeddings_dict),
    'embedding_dim': emb_size
}
metadata_path = os.path.join(model_dir, "metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata to: {metadata_path}")

# Also save embeddings to training_data for next notebooks
embeddings_df.to_csv(os.path.join(TRAINING_DATA_PATH, "node_embeddings.csv"), index=False)
print(f"\nAlso saved embeddings to: {TRAINING_DATA_PATH}/node_embeddings.csv")

print(f"\n{'='*50}")
print(f"Model saved to: {model_dir}")
print(f"Accuracy: {accuracy:.4f}")
print(f"{'='*50}")

Saved Node2Vec model to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4/node2vec_model.model
Saved classifier to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4/classifier.joblib
Saved embeddings to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4/node_embeddings.csv
Saved metadata to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4/metadata.json



Also saved embeddings to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/node_embeddings.csv

Model saved to: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_e2366de4
Accuracy: 0.8891
